In [1]:
import pandas as pd
from evidently.report import Report
from evidently.metric_preset import DataDriftPreset

In [2]:
df = pd.read_csv("../data/processed/cleaned_retail.csv")
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 361461 entries, 0 to 361460
Data columns (total 9 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   invoiceno    361461 non-null  int64  
 1   stockcode    361461 non-null  object 
 2   description  361461 non-null  object 
 3   quantity     361461 non-null  int64  
 4   invoicedate  361461 non-null  object 
 5   unitprice    361461 non-null  float64
 6   customerid   361461 non-null  float64
 7   country      361461 non-null  object 
 8   revenue      361461 non-null  float64
dtypes: float64(3), int64(2), object(4)
memory usage: 24.8+ MB


In [3]:
df["invoicedate"] = pd.to_datetime(df["invoicedate"])
df = df.sort_values("invoicedate")

In [4]:
reference_data = df.iloc[:250000]
current_data = df.iloc[250000:]

In [5]:
columns = [
    "quantity",
    "unitprice",
    "revenue",
    "country"
]

reference_data = reference_data[columns]
current_data = current_data[columns]

In [6]:
report = Report(metrics=[DataDriftPreset()])

In [7]:
report.run(reference_data=reference_data,current_data=current_data)

In [8]:
report.save_html("../monitoring/reports/data_drift_report.html")

In [9]:
result = report.as_dict()
print(result)

{'metrics': [{'metric': 'DatasetDriftMetric', 'result': {'drift_share': 0.5, 'number_of_columns': 4, 'number_of_drifted_columns': 1, 'share_of_drifted_columns': 0.25, 'dataset_drift': False}}, {'metric': 'DataDriftTable', 'result': {'number_of_columns': 4, 'number_of_drifted_columns': 1, 'share_of_drifted_columns': 0.25, 'dataset_drift': False, 'drift_by_columns': {'quantity': {'column_name': 'quantity', 'column_type': 'num', 'stattest_name': 'Wasserstein distance (normed)', 'stattest_threshold': 0.1, 'drift_score': 0.07400695055307761, 'drift_detected': False, 'current': {'small_distribution': {'x': [1.0, 29.7, 58.4, 87.1, 115.8, 144.5, 173.2, 201.9, 230.6, 259.3, 288.0], 'y': [0.03402293147519576, 0.0006458408117437493, 0.00010503509813451103, 5.3142757984722826e-05, 9.690738220743573e-06, 6.252089174673277e-07, 2.8134401286029717e-06, 3.1260445873366384e-07, 6.25208917467327e-07, 2.188231211135647e-06]}}, 'reference': {'small_distribution': {'x': [1.0, 90.9, 180.8, 270.7000000000000

In [10]:
drift_detected = result["metrics"][0]["result"]["dataset_drift"]
if drift_detected:
    print("ALERT: Drift detected!")
else:
    print("No significant drift.")

No significant drift.


In [11]:
import json

with open("../monitoring/reports/drift_metrics.json", "w") as f:
    json.dump(result, f)